<a href="https://colab.research.google.com/github/lindy-zhang/my-own-llm/blob/main/llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import urllib.request

# Use Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r") as f:
  text = f.read()

print(f"Length of dataset in characters: {len(text)}")
print("\nFirst 500 characters:\n")
print(text[:500])


Length of dataset in characters: 1115394

First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
# Start building tokenizer

chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocabulary: ", "".join(chars))
print(f"Vocab size: {vocab_size}")

# Lookup tables
# str (char) -> int: encode text to token IDs
# int -> str (char): decoding token IDs back to text
str_int = {ch: i for i, ch in enumerate(chars)}
int_str = {i: ch for i, ch in enumerate(chars)}

def encode(s):
  """
  Convert a string into a list of integer token IDs
  """
  return [str_int[c] for c in s]

def decode(ids):
  """
  Convert a list of integer token IDs into a string
  """
  return "".join([int_str[i] for i in ids])

# Test
print(encode("hello"))
print(decode(encode("hello")))

Vocabulary:  
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65
[46, 43, 50, 50, 53]
hello


In [ ]:
# Prep data (split into dataset into train/validation)
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print("Full dataset shape", data.shape, "dtype:", data.dtype)

# Split dataset - 90% train, 10% validation
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train size: {len(train_data)} tokens")
print(f"Validation size: {len(val_data)} tokens")



Full dataset shape torch.Size([1115394]) dtype: torch.int64
Train size: 1003854 tokens
Validation size: 111540 tokens


In [ ]:
block_size = 256 # num chars of context the model sees (*might change later)

# Training instance example
x = train_data[:block_size]
y = train_data[1:block_size+1]

print("Example input (first 20 tokens):", x[:20].tolist())
print("\nExample target (first 20 tokens):", y[:20].tolist())
print("\nAs text -- input: ", decode(x[:20].tolist()))
print("As text -- target:", decode(y[:20].tolist()))

Example input (first 20 tokens): [18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56]

Example target (first 20 tokens): [47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43]

As text -- input:  First Citizen:
Befor
As text -- target: irst Citizen:
Before


In [ ]:
# Batching
batch_size = 64 # how many indep. sequences to process in parallel
block_size = 256 # context length (same as above)

def get_batch(split: str):
  """
  Returns a batch of (input, target) pairs, each of shape
  (batch_size, block_size) sampled from random positions in the data

  -> Takes in str argument: "train" or "val"
  """

  data = train_data if split=="train" else val_data # which data to use

  # Generate arr of random ints
  ix = torch.randint(len(data)-block_size-1, (batch_size,))
  # Stack tensors into matrix of shape (64, 256)
  x = torch.stack([data[i: i+block_size] for i in ix])
  # Targets (same as input chunk, shifted 1 to right)
  y = torch.stack([data[i+1: i+block_size+1] for i in ix])

  return x, y

# x = input matrix, y = target matrix
xb, yb = get_batch("train")
print("Input batch shape:", xb.shape)   # (batch_size, block_size)
print("Target batch shape:", yb.shape)  # (batch_size, block_size)

print("\nFirst sequence in the batch, as text:")
print(decode(xb[0].tolist()))


Input batch shape: torch.Size([64, 256])
Target batch shape: torch.Size([64, 256])

First sequence in the batch, as text:
ou may stay;
For I have more to commune with Bianca.

KATHARINA:
Why, and I trust I may go too, may I not? What,
shall I be appointed hours; as though, belike, I
knew not what to take and what to leave, ha?

GREMIO:
You may go to the devil's dam: your gift


In [ ]:
# Self-attention

import torch.nn as nn
import torch.nn.functional as F

n_embd = 384 # size of embedding vector for each token
n_head = 6 # num of attention heads
head_size = n_embd/n_head

class Head(nn.Module):
  """
  This is 1 self-attention head
  """

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)

    self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

  def forward(self, x):
    B, T, C = x.shape  # Batch, Time (sequence length), Channels (n_embd)

        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # Attention scores: for every pair of positions, how much should
        # query i attend to key j? Scaled by sqrt(head_size) -- this
        # scaling prevents the dot products from growing too large as
        # head_size increases, which would make softmax overly peaked.
        wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, T)

        # Causal mask: block attention to future positions by setting
        # their scores to -inf BEFORE softmax, so softmax assigns them
        # exactly zero weight.
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)  # (B, T, T), each row sums to 1

        # Weighted sum of values, using our attention weights
        out = wei @ v  # (B, T, head_size)
        return out



